In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib

def train_absa_model(data_path):
    print("⏳ 1. Đang tải dữ liệu huấn luyện...")
    df = pd.read_csv(data_path)
    
    # Loại bỏ các dòng bị trống nếu có
    df = df.dropna()
    
    X = df['text_with_aspect'] # Văn bản đầu vào đã gộp khía cạnh
    y = df['sentiment']        # Nhãn cảm xúc (Positive, Negative, Neutral)
    
    # 2. Chia tập dữ liệu thành 80% Huấn luyện (Train) và 20% Kiểm thử (Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"📊 Tập Train: {len(X_train)} dòng | Tập Test: {len(X_test)} dòng")
    
    # 3. Xây dựng Pipeline: Biến đổi chữ thành số (TF-IDF) -> Đưa vào mô hình SVM (LinearSVC)
    # Chúng ta dùng ngram_range=(1, 3) để mô hình học được các cụm 3 từ (ví dụ: "chụp ảnh đẹp", "pin sụt nhanh")
    print("⚙️ 2. Đang khởi tạo cấu hình mô hình (TF-IDF + SVM)...")
    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 3), 
            max_features=50000, # Giới hạn 50,000 từ phổ biến nhất để tránh tràn bộ nhớ
            sublinear_tf=True
        )),
        ('svm', LinearSVC(C=1.0, class_weight='balanced', random_state=42)) 
        # class_weight='balanced' giúp xử lý vấn đề lệch nhãn (nhiều Positive, ít Negative)
    ])
    
    # 4. Bắt đầu huấn luyện mô hình
    print("🚀 3. Đang huấn luyện mô hình trên CPU (Vui lòng đợi vài giây)...")
    model_pipeline.fit(X_train, y_train)
    print("✅ Huấn luyện hoàn tất!")
    
    # 5. Đánh giá mô hình trên tập dữ liệu Test độc lập
    print("\n📊 4. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH:")
    y_pred = model_pipeline.predict(X_test)
    
    print(f"🎯 Độ chính xác tổng thể (Accuracy): {accuracy_score(y_test, y_pred):.2%}")
    print("\n📋 Báo cáo chi tiết theo từng nhãn cảm xúc (Classification Report):")
    print(classification_report(y_test, y_pred))
    
    # 6. Lưu mô hình đã huấn luyện thành file để tái sử dụng sau này
    model_filename = 'absa_svm_model.pkl'
    joblib.dump(model_pipeline, model_filename)
    print(f"💾 Đã lưu mô hình thành công vào file '{model_filename}'")
    
    return model_pipeline

if __name__ == "__main__":
    # Tiến hành chạy huấn luyện
    trained_model = train_absa_model('data_machine_learning.csv')
    
    # --- CHẠY THỬ NGHIỆM DỰ ĐOÁN CÂU MỚI ---
    print("\n🔮 5. CHẠY THỬ NGHIỆM DỰ ĐOÁN THỰC TẾ:")
    
    # Giả sử người dùng nhập câu này
    test_comment = "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
    
    # Danh sách các khía cạnh hệ thống muốn kiểm tra trong câu trên
    aspects_to_test = ["FEATURES", "CAMERA", "BATTERY", "PRICE"]
    
    print(f"💬 Câu đánh giá: \"{test_comment}\"")
    print("-" * 50)
    
    for aspect in list(aspects_to_test):
        # Định dạng đúng cấu trúc đã học: Câu_văn [ASPECT] Tên_Khía_Cạnh
        input_format = f"{test_comment} [ASPECT] {aspect}"
        
        # Dự đoán
        prediction = trained_model.predict([input_format])[0]
        print(f"👉 Khía cạnh [{aspect}]: {prediction}")


⏳ 1. Đang tải dữ liệu huấn luyện...
📊 Tập Train: 19097 dòng | Tập Test: 4775 dòng
⚙️ 2. Đang khởi tạo cấu hình mô hình (TF-IDF + SVM)...
🚀 3. Đang huấn luyện mô hình trên CPU (Vui lòng đợi vài giây)...
✅ Huấn luyện hoàn tất!

📊 4. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH:
🎯 Độ chính xác tổng thể (Accuracy): 73.70%

📋 Báo cáo chi tiết theo từng nhãn cảm xúc (Classification Report):
              precision    recall  f1-score   support

    Negative       0.75      0.73      0.74      1493
     Neutral       0.35      0.40      0.37       581
    Positive       0.82      0.81      0.82      2701

    accuracy                           0.74      4775
   macro avg       0.64      0.65      0.64      4775
weighted avg       0.74      0.74      0.74      4775

💾 Đã lưu mô hình thành công vào file 'absa_svm_model.pkl'

🔮 5. CHẠY THỬ NGHIỆM DỰ ĐOÁN THỰC TẾ:
💬 Câu đánh giá: "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
-----------------

In [2]:
import pandas as pd

def print_dataset_aspects(ml_csv_path):
    print(f"⏳ Đang đọc file '{ml_csv_path}'...")
    df = pd.read_csv(ml_csv_path)
    
    # Hàm này dùng để tách lấy tên Khía cạnh đứng sau token [ASPECT]
    def extract_aspect(text):
        if "[ASPECT]" in text:
            return text.split("[ASPECT]")[-1].strip()
        return "UNKNOWN"
    
    # Tạo một cột mới chứa riêng tên khía cạnh
    df['aspect_name'] = df['text_with_aspect'].apply(extract_aspect)
    
    # 1. Lấy danh sách các khía cạnh duy nhất
    unique_aspects = df['aspect_name'].unique()
    print("\n==================================================")
    print(f"📋 DANH SÁCH TẤT CẢ CÁC ASPECT ĐANG CÓ ({len(unique_aspects)} khía cạnh):")
    print("==================================================")
    for idx, aspect in enumerate(unique_aspects, 1):
        print(f"{idx}. {aspect}")
        
    # 2. Thống kê số lượng dòng dữ liệu của từng khía cạnh
    print("\n==================================================")
    print("📊 THỐNG KÊ SỐ LƯỢNG DÒNG THEO TỪNG ASPECT:")
    print("==================================================")
    aspect_counts = df['aspect_name'].value_counts()
    print(aspect_counts.to_string())
    
    # 3. Thống kê ma trận chéo giữa Khía cạnh và Sắc thái cảm xúc (Rất quan trọng)
    print("\n==================================================")
    print("⚖️ PHÂN BỔ SẮC THÁI (SENTIMENT) TRÊN TỪNG ASPECT:")
    print("==================================================")
    # Bảng này sẽ cho bạn thấy rõ khía cạnh nào bị thiếu nhãn Neutral hoặc Negative
    cross_tab = pd.crosstab(df['aspect_name'], df['sentiment'])
    print(cross_tab)

# Chạy thống kê trên file dữ liệu Machine Learning bẻ nhỏ của bạn
print_dataset_aspects('data_machine_learning.csv')


⏳ Đang đọc file 'data_machine_learning.csv'...

📋 DANH SÁCH TẤT CẢ CÁC ASPECT ĐANG CÓ (10 khía cạnh):
1. CAMERA
2. FEATURES
3. BATTERY
4. PRICE
5. GENERAL
6. SER&ACC
7. PERFORMANCE
8. SCREEN
9. DESIGN
10. STORAGE

📊 THỐNG KÊ SỐ LƯỢNG DÒNG THEO TỪNG ASPECT:
aspect_name
GENERAL        4866
PERFORMANCE    4140
BATTERY        3604
FEATURES       2642
CAMERA         2146
PRICE          2061
SER&ACC        1995
DESIGN         1378
SCREEN          949
STORAGE          91

⚖️ PHÂN BỔ SẮC THÁI (SENTIMENT) TRÊN TỪNG ASPECT:
sentiment    Negative  Neutral  Positive
aspect_name                             
BATTERY          1228      349      2027
CAMERA            627      288      1231
DESIGN            302       77       999
FEATURES         1659      198       785
GENERAL           949      290      3627
PERFORMANCE      1496      391      2253
PRICE             316     1136       609
SCREEN            379       56       514
SER&ACC           487      107      1401
STORAGE            21       1

In [3]:
import joblib
import re

# 1. Định nghĩa từ điển từ khóa để nhận diện vị trí khía cạnh trong câu văn tiếng Việt
ASPECT_KEYWORDS = {
    "CAMERA": ["camera", "chụp ảnh", "quay phim", "selfie", "ống kính", "hình ảnh", "chụp", "quay"],
    "BATTERY": ["pin", "bin", "sạc", "sac", "dung lượng", "trâu"],
    "PRICE": ["giá", "tiền", "giá thành", "chát", "đắt", "rẻ", "túi tiền", "mua"],
    "SCREEN": ["màn hình", "màn", "hiển thị", "màu sắc", "screen"],
    "PERFORMANCE": ["mượt", "lag", "đơ", "chơi game", "hiệu năng", "cấu hình", "chip", "tốc độ", "load"],
    "FEATURES": ["loa", "wifi", "wf", "sóng", "vân tay", "khu khuôn mặt", "bluetooth", "tính năng", "chức năng"],
    "DESIGN": ["thiết kế", "đẹp", "màu", "vỏ", "sang trọng", "ngoại hình", "cầm"],
    "SER&ACC": ["nhân viên", "tư vấn", "phục vụ", "thegioididong", "tgdd", "giao hàng", "bảo hành", "nhiệt tình"],
    "STORAGE": ["bộ nhớ", "dung lượng", "rom", "ram", "lưu trữ", "gb"],
    "GENERAL": ["máy", "điện thoại", "sản phẩm", "ok", "tốt", "dùng", "xài"]
}

def get_context_window(text, aspect_name, window_size=6):
    """
    Hàm thông minh: Cắt lấy đoạn chữ xung quanh từ khóa của khía cạnh để mô hình ML không bị rối.
    """
    words = text.lower().split()
    keywords = ASPECT_KEYWORDS.get(aspect_name, [])
    
    # Tìm vị trí của từ khóa khía cạnh trong câu
    for idx, word in enumerate(words):
        # Làm sạch từ để so sánh (xóa dấu phẩy, dấu chấm)
        clean_word = re.sub(r'[^\w\s]', '', word)
        if any(kw in clean_word for kw in keywords):
            # Cắt lấy các từ xung quanh vị trí từ khóa tìm thấy
            start = max(0, idx - window_size)
            end = min(len(words), idx + window_size + 1)
            context = " ".join(words[start:end])
            return context
            
    # Nếu không tìm thấy từ khóa cụ thể nào, trả về toàn bộ câu dạng chữ thường
    return text.lower()

def predict_absa_smart(model_path, text_comment):
    # Tải mô hình SVM (.pkl) đã huấn luyện ở bước trước lên
    try:
        model = joblib.load(model_path)
    except FileNotFoundError:
        print(f"❌ Không tìm thấy file mô hình '{model_path}'. Bạn cần chạy file huấn luyện trước!")
        return

    print(f"\n💬 Câu đánh giá gốc: \"{text_comment}\"")
    print("=" * 60)
    print(f"{'KHÍA CẠNH':<15} | {'ĐOẠN NGỮ CẢNH ĐƯỢC CẮT RA':<45} | {'DỰ ĐOÁN'}")
    print("=" * 60)
    
    # Duyệt qua tất cả 10 khía cạnh của bạn để dự đoán
    for aspect in ASPECT_KEYWORDS.keys():
        # Bước 1: Trích xuất vùng chữ có chứa khía cạnh
        context_text = get_context_window(text_comment, aspect)
        
        # Bước 2: Ép đúng định dạng cấu trúc mà mô hình SVM đã được học
        input_format = f"{context_text} [ASPECT] {aspect}"
        
        # Bước 3: Đưa vào mô hình dự đoán sắc thái
        pred_sentiment = model.predict([input_format])[0]
        
        # Chỉ in ra màn hình các khía cạnh thực sự xuất hiện trong ngữ cảnh câu văn
        if context_text != text_comment.lower() or aspect == "GENERAL":
            # Rút ngắn chuỗi in ra cho đẹp bảng
            display_context = context_text if len(context_text) < 45 else context_text[:42] + "..."
            print(f"{aspect:<15} | {display_context:<45} | {pred_sentiment}")

# --- CHẠY THỬ NGHIỆM THỰC TẾ ---
if __name__ == "__main__":
    test_sentence = "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
    
    predict_absa_smart('absa_svm_model.pkl', test_sentence)



💬 Câu đánh giá gốc: "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
KHÍA CẠNH       | ĐOẠN NGỮ CẢNH ĐƯỢC CẮT RA                     | DỰ ĐOÁN
BATTERY         | hình quá đỉnh luôn nhưng cục sạc nhanh đi ... | Negative
PRICE           | đi kèm nóng máy kinh khủng, giá hơi chát.     | Neutral
SCREEN          | điện thoại này xài mượt, màn hình quá đỉnh... | Negative
PERFORMANCE     | điện thoại này xài mượt, màn hình quá đỉnh... | Positive
GENERAL         | điện thoại này xài mượt, màn hình quá đỉnh... | Positive


In [4]:
import joblib
import re

ASPECT_KEYWORDS = {
    "CAMERA": ["camera", "chụp ảnh", "quay phim", "selfie", "ống kính", "hình ảnh", "chụp", "quay"],
    "BATTERY": ["pin", "bin", "sạc", "sac", "dung lượng", "trâu"],
    "PRICE": ["giá", "tiền", "giá thành", "chát", "đắt", "rẻ", "túi tiền", "mua"],
    "SCREEN": ["màn hình", "màn", "hiển thị", "màu sắc", "screen"],
    "PERFORMANCE": ["mượt", "lag", "đơ", "chơi game", "hiệu năng", "cấu hình", "chip", "tốc độ", "load"],
    "FEATURES": ["loa", "wifi", "wf", "sóng", "vân tay", "khu khuôn mặt", "bluetooth", "tính năng", "chức năng"],
    "DESIGN": ["thiết kế", "đẹp", "màu", "vỏ", "sang trọng", "ngoại hình", "cầm"],
    "SER&ACC": ["nhân viên", "tư vấn", "phục vụ", "thegioididong", "tgdd", "giao hàng", "bảo hành", "nhiệt tình"],
    "STORAGE": ["bộ nhớ", "dung lượng", "rom", "ram", "lưu trữ", "gb"],
    "GENERAL": ["máy", "điện thoại", "sản phẩm", "ok", "tốt", "dùng", "xài"]
}

def clean_text(text):
    """
    Hàm làm sạch chuẩn hóa: Thêm khoảng trắng vào trước/sau dấu câu để không bị dính chữ (ví dụ: 'mượt, màn' -> 'mượt , màn')
    """
    text = text.lower()
    # Thêm khoảng trắng xung quanh các dấu câu thông dụng
    text = re.sub(r'([,.\?!\(\)\-\"\'])', r' \1 ', text)
    # Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_context_window(cleaned_text, aspect_name, window_size=5):
    """
    Cắt ngữ cảnh dựa trên văn bản đã được làm sạch dấu câu.
    """
    words = cleaned_text.split()
    keywords = ASPECT_KEYWORDS.get(aspect_name, [])
    
    for idx, word in enumerate(words):
        if any(kw == word for kw in keywords) or any(kw in word for kw in keywords):
            start = max(0, idx - window_size)
            end = min(len(words), idx + window_size + 1)
            return " ".join(words[start:end])
            
    return cleaned_text

def predict_absa_final(model_path, text_comment):
    try:
        model = joblib.load(model_path)
    except FileNotFoundError:
        print(f"❌ Không tìm thấy file mô hình '{model_path}'.")
        return

    # Tiền xử lý câu văn trước khi cắt ngữ cảnh
    cleaned_text = clean_text(text_comment)

    print(f"\n💬 Câu đánh giá gốc: \"{text_comment}\"")
    print("=" * 70)
    print(f"{'KHÍA CẠNH':<15} | {'ĐOẠN NGỮ CẢNH ĐƯỢC CẮT RA':<45} | {'DỰ ĐOÁN'}")
    print("=" * 70)
    
    for aspect in ASPECT_KEYWORDS.keys():
        context_text = get_context_window(cleaned_text, aspect)
        
        # Ép cấu trúc huấn luyện
        input_format = f"{context_text} [ASPECT] {aspect}"
        pred_sentiment = model.predict([input_format])
        
        # Chỉ hiển thị khía cạnh nếu tìm thấy ngữ cảnh thu hẹp (hoặc khía cạnh chung GENERAL)
        if context_text != cleaned_text or aspect == "GENERAL":
            display_context = context_text if len(context_text) < 45 else context_text[:42] + "..."
            print(f"{aspect:<15} | {display_context:<45} | {pred_sentiment}")

if __name__ == "__main__":
    test_sentence = "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
    predict_absa_final('absa_svm_model.pkl', test_sentence)



💬 Câu đánh giá gốc: "Điện thoại này xài mượt, màn hình quá đỉnh luôn nhưng cục sạc nhanh đi kèm nóng máy kinh khủng, giá hơi chát."
KHÍA CẠNH       | ĐOẠN NGỮ CẢNH ĐƯỢC CẮT RA                     | DỰ ĐOÁN
BATTERY         | quá đỉnh luôn nhưng cục sạc nhanh đi kèm n... | ['Negative']
PRICE           | nóng máy kinh khủng , giá hơi chát .          | ['Neutral']
SCREEN          | thoại này xài mượt , màn hình quá đỉnh luô... | ['Positive']
PERFORMANCE     | điện thoại này xài mượt , màn hình quá đỉnh   | ['Positive']
GENERAL         | điện thoại này xài mượt , màn hình quá        | ['Positive']
